# Tarea 2 · Parte 2 — Evaluación, comparación de modelos y extracción estructurada

**Estudiante:** Sandy Patricia Martinez Jara (@sandypmj)

**Reglas para métricas válidas:** la muestra de evaluación (10 documentos y 10 clips del corpus de la Parte 1) se sortea con semilla fija y se guarda en `datos/muestra_evaluacion.csv`. Las configuraciones de **todos** los modelos se fijan antes de calcular métricas y no se cambian después.

In [1]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import utiles, voz
warnings.filterwarnings("ignore")

RAW = Path("datos/raw")
SEMILLA = 2026
docs = pd.read_pickle(RAW / "docs_ocr_respaldo.pkl")
aud = pd.read_pickle(RAW / "aud_respaldo.pkl")
aud["fuente_corta"] = aud["fuente"].str.split(" \\(").str[0]

MUESTRA = Path("datos/muestra_evaluacion.csv")
if MUESTRA.exists():
    muestra = pd.read_csv(MUESTRA)
else:
    cupos = {"impreso": 4, "foto": 3, "manuscrito": 3}     # los tres tipos
    m_doc = pd.concat([docs[docs["tipo"] == t].sample(n, random_state=SEMILLA) for t, n in cupos.items()])
    b = aud.sample(frac=1, random_state=SEMILLA)
    b["turno"] = b.groupby("fuente_corta").cumcount()      # alterna fuentes: entrevistas con ruido y lectura limpia
    m_aud = b.sort_values(["turno", "fuente_corta"]).head(10)
    muestra = pd.concat([pd.DataFrame({"id": m_doc["id"], "modalidad": "documento"}),
                         pd.DataFrame({"id": m_aud["id"], "modalidad": "audio"})])
    muestra.to_csv(MUESTRA, index=False)

ids_doc = muestra.query("modalidad == 'documento'")["id"].tolist()
ids_aud = muestra.query("modalidad == 'audio'")["id"].tolist()
ev_doc = docs.set_index("id").loc[ids_doc]
ev_aud = aud.set_index("id").loc[ids_aud]
display(ev_doc[["tipo", "descripcion"]])
display(ev_aud[["fuente_corta", "condicion", "duracion_seg"]])

,tipo,descripcion
id,,
doc_005,impreso,Billete de 100 intis (1987)
doc_006,impreso,Billete de 1000 intis (1988)
doc_004,impreso,Billete de 50 intis (1987)
doc_008,impreso,Billete de 10 000 intis (1988)
doc_017,foto,Captura propia: foto_03
doc_018,foto,Captura propia: foto_04
doc_016,foto,Captura propia: foto_02
doc_024,manuscrito,Captura propia: manuscrito_05
doc_020,manuscrito,Captura propia: manuscrito_01


,fuente_corta,condicion,duracion_seg
id,,,
clip_012,OpenSLR 73,lectura en estudio (limpio),66.2
clip_004,Wikimedia Commons,entrevista en exteriores / con ruido,89.7
clip_009,OpenSLR 73,lectura en estudio (limpio),60.3
clip_005,Wikimedia Commons,entrevista en exteriores / con ruido,9.9
clip_016,OpenSLR 73,lectura en estudio (limpio),61.4
clip_008,Wikimedia Commons,entrevista en exteriores / con ruido,80.9
clip_023,OpenSLR 73,lectura en estudio (limpio),63.4
clip_001,Wikimedia Commons,entrevista en exteriores / con ruido,79.7
clip_028,OpenSLR 73,lectura en estudio (limpio),64.8


## Modelos alternativos (configuración fijada antes de ver métricas)

- **Documentos: EasyOCR** (`es`, `paragraph=True`, CPU). Usa otra arquitectura (detector CRAFT + reconocedor CRNN) y se instala solo con pip.
- **Audio: Parakeet TDT 0.6B v3** (NVIDIA), vía `voz.Transcriptor`, con la misma segmentación por VAD. No es un modelo generativo, así que no "inventa" frases en el silencio. Si no carga en esta PC, se usa **Qwen3-ASR 0.6B** (decisión tomada de antemano).
- Las salidas y los tiempos se guardan en `datos/raw/alternativos.json`, caso por caso (si se corta, continúa donde quedó).

In [3]:
import easyocr

ALT = RAW / "alternativos.json"
alt = json.loads(ALT.read_text(encoding="utf-8")) if ALT.exists() else {}
def guardar():
    ALT.write_text(json.dumps(alt, ensure_ascii=False), encoding="utf-8")

# --- Documentos: EasyOCR ---
lector = easyocr.Reader(["es"], gpu=False)
for i, r in ev_doc.iterrows():
    k = f"easyocr|{i}"
    if k in alt:
        continue
    t0 = time.perf_counter()
    texto = " ".join(lector.readtext(r["ruta_original"], detail=0, paragraph=True))
    alt[k] = {"texto": texto, "segundos": round(time.perf_counter() - t0, 2), "modelo": "EasyOCR (es, paragraph=True, CPU)"}
    guardar()
    print(f"{k}: {alt[k]['segundos']} s")

# --- Audio: Parakeet (o Qwen3-ASR si Parakeet no carga) ---
asr_alt = None
for clave in ["parakeet-v3", "qwen3-asr-0.6b"]:
    try:
        asr_alt = voz.Transcriptor(clave)
        break
    except Exception as e:
        print(f"No se pudo cargar {clave}: {str(e)[:150]}")
nombre_alt = voz.MODELOS[asr_alt.clave]["nombre"]
print("ASR alternativo:", nombre_alt, "en", asr_alt.device)

for i, r in ev_aud.iterrows():
    k = f"asr_alt|{i}"
    if k in alt:
        continue
    x = voz.cargar_audio(r["archivo_audio"])
    t0 = time.perf_counter()
    texto = " ".join(s.texto for s in asr_alt.segmentos(x, idioma="es"))
    alt[k] = {"texto": texto, "segundos": round(time.perf_counter() - t0, 2), "modelo": nombre_alt}
    guardar()
    print(f"{k}: {alt[k]['segundos']} s")

print("\n✅ Modelos alternativos listos:", len(alt), "resultados guardados")

Using CPU. Note: This module is much faster with a GPU.


No se pudo cargar parakeet-v3: 
ParakeetFeatureExtractor requires the librosa library. But that was not found in your environment. You can install them with pip:
`pip install libros


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

ASR alternativo: Qwen3-ASR 0.6B en cpu

✅ Modelos alternativos listos: 20 resultados guardados
